# 🤖 JINXMP3 AI-OS Brain — Colab Enterprise
**Runs in your Google Cloud project `verdant-medium-500508-i4`.**
No disconnects. Direct GCS access. A100 GPU.

This notebook becomes the AI brain that controls your entire laptop.

### Setup:
1. Run all cells top to bottom
2. Copy the public URL from the last cell output
3. On your laptop: `export COLAB_URL=<url> && python main.py --backend colab --both`

In [ ]:
# Cell 1 — Install
!pip install flask pyngrok transformers accelerate bitsandbytes anthropic google-cloud-storage -q
!nvidia-smi

In [ ]:
# Cell 2 — Authenticate (already done in Enterprise Colab)
import google.auth
credentials, project = google.auth.default()
print(f'Project: {project}')
print(f'Credentials: {type(credentials).__name__}')

GCP_PROJECT = 'verdant-medium-500508-i4'
GCS_BUCKET  = 'jinxmp3-assets'
NGROK_TOKEN = ''  # Optional: ngrok.com/signup for stable URL

# Model choice — Enterprise has A100 so we can run bigger models
MODEL_ID = 'mistralai/Mistral-7B-Instruct-v0.3'  # Best free option on A100
# MODEL_ID = 'meta-llama/Llama-3.1-8B-Instruct'  # Needs HF token
# MODEL_ID = 'microsoft/Phi-3.5-mini-instruct'    # Fastest
print(f'Model: {MODEL_ID}')

In [ ]:
# Cell 3 — Load AI model
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM, pipeline

print(f'GPU: {torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU"}')
print(f'Loading {MODEL_ID}...')

tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, trust_remote_code=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    torch_dtype=torch.float16,
    device_map='auto',
    load_in_4bit=True,
    trust_remote_code=True
)
pipe = pipeline(
    'text-generation', model=model, tokenizer=tokenizer,
    max_new_tokens=2048, temperature=0.3, do_sample=True,
    return_full_text=False
)
print('✅ Model ready!')

In [ ]:
# Cell 4 — Start AI-OS Brain Server
from flask import Flask, request, jsonify
from threading import Thread
from pyngrok import ngrok
from google.cloud import storage
import json, os

app = Flask('ai-os-brain')

def to_prompt(messages):
    """Convert messages to Mistral/Phi chat format."""
    out = ''
    for m in messages:
        r, c = m.get('role'), m.get('content', '')
        if r == 'system':    out += f'[INST] <<SYS>>\n{c}\n<</SYS>>\n\n'
        elif r == 'user':    out += f'{c} [/INST] '
        elif r == 'assistant': out += f'{c} </s><s>[INST] '
    return out

@app.get('/health')
def health():
    return jsonify({
        'status': 'ok',
        'model': MODEL_ID,
        'project': GCP_PROJECT,
        'gpu': torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'cpu'
    })

@app.post('/chat')
def chat():
    try:
        data = request.json
        messages = data.get('messages', [])
        result = pipe(to_prompt(messages))
        return jsonify({'response': result[0]['generated_text'].strip()})
    except Exception as e:
        return jsonify({'error': str(e)}), 500

# GCS file operations (direct access since in same project)
@app.post('/gcs/upload')
def gcs_upload():
    data = request.json
    client = storage.Client(project=GCP_PROJECT)
    bucket = client.bucket(GCS_BUCKET)
    blob = bucket.blob(data['path'])
    blob.upload_from_string(data['content'])
    return jsonify({'ok': True, 'path': f'gs://{GCS_BUCKET}/{data["path"]}'})

@app.get('/gcs/list')
def gcs_list():
    client = storage.Client(project=GCP_PROJECT)
    blobs = list(client.list_blobs(GCS_BUCKET))
    return jsonify({'files': [b.name for b in blobs]})

# Start server
Thread(target=lambda: app.run(port=5000, use_reloader=False), daemon=True).start()

# ngrok tunnel
if NGROK_TOKEN:
    ngrok.set_auth_token(NGROK_TOKEN)
tunnel = ngrok.connect(5000)
PUBLIC_URL = tunnel.public_url

print()
print('=' * 65)
print('  JINXMP3 AI-OS ENTERPRISE BRAIN — LIVE')
print(f'  Model : {MODEL_ID}')
print(f'  GPU   : {torch.cuda.get_device_name(0)}')
print(f'  URL   : {PUBLIC_URL}')
print('=' * 65)
print()
print('Run on your laptop:')
print(f'  export COLAB_URL={PUBLIC_URL}')
print(f'  cd ~/Ai-operating-system && source .venv/bin/activate')
print(f'  python main.py --backend colab --both')
print()
print('Your entire computer is now powered by Google Enterprise GPU!')

In [ ]:
# Cell 5 — Keep Alive (run this to prevent timeout)
import time, requests
print('Keeping alive... GPU is running. Ctrl+C to stop.')
while True:
    r = requests.get('http://localhost:5000/health').json()
    print(f"[{time.strftime('%H:%M:%S')}] Status: {r['status']} | GPU: {r.get('gpu','?')}")
    time.sleep(60)